In [2]:
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from trajectory_plotter import TrajectoryValidationRiskPloter

In [1]:
print ("hey")

hey


In [ ]:
# --- Initial run ---
plotter = TrajectoryValidationRiskPloter(max_T=500)
plotter.X, plotter.y = plotter.generate_data(n=200, p=400, random_seed=42)
X_train, y_train, X_valid, y_valid = plotter.split_train_valid()
plotter.w_traj, plotter.train_over_time, plotter.valid_over_time = plotter.run_GD(
    X_train, y_train, X_valid, y_valid, eta=0.1
)
print(f"Ran {len(plotter.train_over_time)} iterations.")

In [ ]:
# --- Build figure ---
ts = list(range(1, len(plotter.train_over_time) + 1))

fig = go.FigureWidget()
fig.add_scatter(x=ts, y=plotter.train_over_time, name='Train Risk',      line=dict(color='steelblue'))
fig.add_scatter(x=ts, y=plotter.valid_over_time, name='Validation Risk', line=dict(color='tomato'))
fig.update_layout(
    xaxis_title='Iteration t',
    yaxis_title='MSE',
    title='GD Trajectory',
    height=450,
    legend=dict(x=0.75, y=0.95),
)

# --- Scale toggles ---
x_scale = widgets.ToggleButtons(
    options=['linear', 'log'], value='linear',
    description='X scale:', button_style=''
)
y_scale = widgets.ToggleButtons(
    options=['linear', 'log'], value='linear',
    description='Y scale:', button_style=''
)

def update_scales(change):
    fig.update_layout(xaxis_type=x_scale.value, yaxis_type=y_scale.value)

x_scale.observe(update_scales, names='value')
y_scale.observe(update_scales, names='value')

# --- Parameter inputs ---
def parse_int(s):
    s = str(s).strip().lower()
    if s.endswith('k'):
        return int(float(s[:-1]) * 1000)
    return int(float(s))  # handles 6e3, 6000, 6k, etc.

w = '180px'
n_input    = widgets.Text(value='200',  description='n:',     layout=widgets.Layout(width=w))
p_input    = widgets.Text(value='400',  description='p:',     layout=widgets.Layout(width=w))
seed_input = widgets.Text(value='42',   description='seed:',  layout=widgets.Layout(width=w))
eta_input  = widgets.Text(value='0.1',  description='eta:',   layout=widgets.Layout(width=w))
maxT_input = widgets.Text(value='500',  description='max_T:', layout=widgets.Layout(width=w))

run_btn = widgets.Button(description='Rerun', button_style='primary', layout=widgets.Layout(width='80px'))
status  = widgets.Label(value='')

def on_run(b):
    status.value = 'Running...'
    run_btn.disabled = True
    try:
        plotter.max_T = parse_int(maxT_input.value)
        plotter.X, plotter.y = plotter.generate_data(
            n=parse_int(n_input.value), p=parse_int(p_input.value),
            random_seed=parse_int(seed_input.value)
        )
        X_tr, y_tr, X_val, y_val = plotter.split_train_valid()
        plotter.w_traj, plotter.train_over_time, plotter.valid_over_time = plotter.run_GD(
            X_tr, y_tr, X_val, y_val, eta=float(eta_input.value)
        )
        new_ts = list(range(1, len(plotter.train_over_time) + 1))
        with fig.batch_update():
            fig.data[0].x = new_ts;  fig.data[0].y = plotter.train_over_time
            fig.data[1].x = new_ts;  fig.data[1].y = plotter.valid_over_time
        status.value = f'Done. ({len(plotter.train_over_time)} iters)'
    except Exception as e:
        status.value = f'Error: {e}'
    run_btn.disabled = False

run_btn.on_click(on_run)

# --- Layout ---
display(widgets.VBox([
    widgets.HBox([x_scale, y_scale]),
    fig,
    widgets.HTML('<hr style="margin:6px 0">'),
    widgets.HBox([n_input, p_input, seed_input, eta_input, maxT_input]),
    widgets.HBox([run_btn, status]),
]))